## Merging Positonal and Token Embeddings together

- Our Input `` X `` is created after combining the Positional Encoding and the Token Embedding ``X = token_emb + pos_emb``

### Implementation Note ``overloading``

- Overloading is when a function has multiple functionalities "giving it more responsibilities then it was initially defined for". 
- Here we ``Overload`` the ``+`` operator over two pieces of data that aren't of the same dimension (but in a specific way). 
- The overload is summarised here ``(B, T, C) + (T, C) = (B, T, C)``
- The ``+`` operator notices ``(T,C)`` then ``stretches`` it to ``(1, T, C )`` then ``adds element wise B`` times

### Implementation Note ``forward()``
- This is where you write the logic of the model but not where you implement it.
- You will define ``forward`` but **never** call ``forward``
- nn.Module ``overload`` ``__call__`` when writing ``t(x)``
- ``__call__`` is what keeps track of {hooks, autograd, train/eval state}

#### How do I know when to use? 

1. "Am I defining what this module does, or using it?"
   - Defining → you're writing the forward method body.
   - Using → you call the object: ``module(x)``.

2. "Is this a nn.Module?" 
   - If yes, the ``object(x)`` convention applies. This includes:
     - Built-in layers: ``nn.Embedding``, ``nn.Linear``, ``nn.LayerNorm`` 
     - Your own modules: ``TokenEmbedding``, ``Embeddings``
     - Loss functions that are modules: ``nn.CrossEntropyLoss()``

Note: we already call these as ``wte(idx)``, never ``wte.forward(idx)``.

#### General framework of forward

```python 
class Block(nn.Module):
    def forward(self, x):          # ← the ONLY method PyTorch auto-calls
        a = self.step_one(x)       # you call your helper, explicitly
        b = self.step_two(a)       # you call your helper, explicitly
        return b

    def step_one(self, x):         # helper — PyTorch never calls this itself
        return x + 1

    def step_two(self, x):         # helper — PyTorch never calls this itself
        return x * 2
```

#### In transformer forward looks like

```python
class Block(nn.Module):
    def forward(self, x):          # WHAT the module does — the whole flow
        x = self.attention(x)      # step named clearly
        x = self.feed_forward(x)   # step named clearly
        return x

    def attention(self, x):        # HOW one step works — detail hidden here
        ...
    def feed_forward(self, x):     # HOW another step works
        ...
```

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class Embeddings(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)

    def forward(self, idx):
        embedded_input = self.wte(idx)
        B, T = idx.shape
        pos = torch.arange(T)
        pos_emb = self.wpe(pos)
        x = embedded_input + pos_emb
        return x